# Get To Know A Dataset: FathomNet

This notebook is a guided tour of the [FathomNet](https://registry.opendata.aws/fathomnet/) dataset on the
Registry of Open Data on AWS. FathomNet is a global, open-source database of ocean imagery and AI-ready
annotations, run by the [Monterey Bay Aquarium Research Institute (MBARI)](https://www.mbari.org/). More
tutorials and documentation for this and other datasets are at the
[Registry of Open Data on AWS](https://registry.opendata.aws/).

> **Draft.** The S3 bucket is not live yet. The code below targets the proposed layout in
> [docs/bucket-layout.md](docs/bucket-layout.md) and has not been run against it.

### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

Everything lives in one public bucket, `s3://fathomnet-pds` in `us-west-2`. No AWS account is needed. At
the top level:

1. `README.md` and `LICENSE.md`, pointers to the documentation and licensing terms.
2. `images/`, every public FathomNet image as contributed, named `<image_uuid>.<ext>`.
3. `thumbnails/`, a 512 px JPEG of every image, named `<image_uuid>.jpg`.
4. `metadata/`, Parquet tables (`images`, `boxes`, `tags`, `collections`, `image_collections`),
   each partitioned by release date as `release=<YYYY-MM-DD>/`. Releases are complete
   snapshots, so pinning one makes your work reproducible.
5. `exports/`, the same annotations as COCO JSON for tools that expect it.

Image UUIDs are the same ones used by the [FathomNet Database](https://database.fathomnet.org) and its
REST API. Full documentation, including a column-by-column
[data dictionary](docs/data-dictionary.md), is in this repository.

In [ ]:
# This notebook requires the following additional libraries
# (install with your preferred tool, e.g. `uv sync` in this repository, or pip/conda):
#
# boto3 >= 1.38
# duckdb >= 1.3
# pandas >= 2.2
# pillow >= 11.0
# matplotlib >= 3.10

# Built-ins
import csv
import io
import json
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

# Installed libraries
import boto3
import duckdb
import matplotlib.pyplot as plt
from botocore import UNSIGNED
from botocore.config import Config
from matplotlib.patches import Rectangle
from PIL import Image

First we define where the dataset lives and create an S3 client. The bucket is public, so requests
are unsigned.

In [ ]:
BUCKET = "fathomnet-pds"
REGION = "us-west-2"

s3 = boto3.client("s3", region_name=REGION, config=Config(signature_version=UNSIGNED))

# Top-level prefixes and objects
top = s3.list_objects_v2(Bucket=BUCKET, Delimiter="/")
for p in top.get("CommonPrefixes", []):
    print(p["Prefix"])
for obj in top.get("Contents", []):
    print(obj["Key"])

The `metadata/` prefix holds one prefix per table, plus `releases.csv`, which lists every release
with its row counts.

In [ ]:
for p in s3.list_objects_v2(Bucket=BUCKET, Prefix="metadata/", Delimiter="/")["CommonPrefixes"]:
    print(p["Prefix"])

body = s3.get_object(Bucket=BUCKET, Key="metadata/releases.csv")["Body"].read().decode()
releases = list(csv.DictReader(io.StringIO(body)))
for r in releases:
    print(r)

We will pin the most recent release for the rest of this notebook. Each table under that release is
one or more Parquet files.

In [ ]:
RELEASE = max(r["release"] for r in releases)
print("Using release", RELEASE)

resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=f"metadata/boxes/release={RELEASE}/")
for obj in resp["Contents"]:
    print(obj["Key"], f"{obj['Size'] / 1e6:.1f} MB")

Images and thumbnails are flat prefixes keyed by UUID. Listing them is rarely useful (there are
hundreds of thousands); instead, select images from the metadata and fetch them by key.

In [ ]:
resp = s3.list_objects_v2(Bucket=BUCKET, Prefix="images/", MaxKeys=5)
for obj in resp["Contents"]:
    print(obj["Key"], obj["Size"])

### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?

**Images (JPEG, PNG, TIFF).** Stills and video frame grabs from remotely operated vehicles, autonomous
underwater vehicles, drop cameras, divers, and citizen scientists. They are stored exactly as contributed,
so sizes, bit depths, and color balance vary widely; that variety is part of what makes FathomNet a
realistic benchmark. [Pillow](https://pillow.readthedocs.io/) reads all three formats. Check the `sha256`
column if you need to verify a download.

**Metadata (Apache Parquet).** [Parquet](https://parquet.apache.org/) is a columnar format: a query that
needs three columns reads only those three, and row groups let readers skip data that cannot match a
filter. That makes it practical to query FathomNet's metadata straight from S3 without downloading it.
The tables are partitioned with [Hive-style](https://docs.aws.amazon.com/athena/latest/ug/partitions.html)
`release=` prefixes. Good tools:

- [DuckDB](https://duckdb.org/) (used here) or [Polars](https://pola.rs/) for SQL or dataframes on a
  laptop or notebook instance.
- [Amazon Athena](https://aws.amazon.com/athena/) for serverless SQL at scale; see
  [docs/athena.md](docs/athena.md) for table definitions.
- PyArrow, pandas, and Spark all read these files natively.

**Annotations (COCO JSON).** The `exports/` prefix has the same boxes in
[COCO](https://cocodataset.org/#format-data) format for detection frameworks that expect it. Like the
`boxes` table, COCO boxes are `[x, y, width, height]` with the origin at the image's **top-left**. YOLO
uses normalized **center** coordinates, so convert before training a YOLO model.

**Labels.** `boxes.concept` is usually a [WoRMS](https://www.marinespecies.org/) scientific name, at
whatever rank the annotator could identify (species, genus, family, and so on). Labels are free
text. To resolve them to accepted names, synonyms, or lineage (for example, to roll species up to
family), use FathomNet's [WoRMS name server](https://database.fathomnet.org/worms/docs).

**Licenses.** Every image and every annotation has its own [SPDX](https://spdx.org/licenses/) license in
`image_license` and `annotation_license`. Filter on them if your use has license constraints; see
[docs/licensing.md](docs/licensing.md).

### Q: Can you show us an example of downloading and loading data from your dataset?

We will query the Parquet tables in place with DuckDB. Its `httpfs` extension reads from S3; with no
credentials configured, requests are anonymous. We register a view per table for our pinned release.

In [ ]:
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"SET s3_region = '{REGION}';")

for table in ["images", "boxes"]:
    path = f"s3://{BUCKET}/metadata/{table}/release={RELEASE}/*.parquet"
    con.execute(f"CREATE OR REPLACE VIEW {table} AS SELECT * FROM read_parquet('{path}')")

con.sql("DESCRIBE boxes")

How big is this release? Counting rows only touches Parquet footers, so this is fast.

In [ ]:
con.sql("""
    SELECT
        (SELECT count(*) FROM images) AS n_images,
        (SELECT count(*) FROM boxes) AS n_boxes,
        (SELECT count(DISTINCT concept) FROM boxes) AS n_concepts,
        (SELECT count(*) FROM boxes WHERE review_state = 'VERIFIED') AS n_verified_boxes
""")

Now we pick one image with several verified boxes, download it from S3, and draw its annotations.
`s3_key` tells us where each image lives.

In [ ]:
example = (
    con.sql("""
    SELECT i.image_uuid, i.s3_key, i.width, i.height, i.depth_meters, i.image_license
    FROM images i JOIN boxes b USING (image_uuid)
    WHERE b.review_state = 'VERIFIED'
    GROUP BY ALL
    HAVING count(*) BETWEEN 3 AND 10
    LIMIT 1
""")
    .df()
    .iloc[0]
)
example

In [ ]:
example_boxes = con.execute(
    "SELECT concept, x, y, width, height FROM boxes WHERE image_uuid = ?",
    [example.image_uuid],
).df()

data = s3.get_object(Bucket=BUCKET, Key=example.s3_key)["Body"].read()
img = Image.open(io.BytesIO(data))

fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(img)
for b in example_boxes.itertuples():
    ax.add_patch(
        Rectangle((b.x, b.y), b.width, b.height, fill=False, edgecolor="yellow", linewidth=2)
    )
    ax.text(b.x, b.y - 5, b.concept, color="yellow", fontsize=10, backgroundcolor="black")
ax.set_title(f"{example.image_uuid}  ({example.depth_meters} m, {example.image_license})")
ax.axis("off")
plt.show()

### Q: A picture is worth a thousand words. Show us a visual (or several!) from your dataset that either illustrates something informative about your dataset, or that you think might excite someone to dig in further.

First, a sample of what FathomNet labels look like. We pick the 12 most-annotated labels, take the
largest verified, unoccluded box of each, and show it on the image's thumbnail. Thumbnails are scaled
copies, so we scale box coordinates to match.

In [ ]:
gallery = con.sql("""
    WITH top_concepts AS (
        SELECT concept
        FROM boxes
        GROUP BY concept
        ORDER BY count(*) DESC
        LIMIT 12
    ),
    candidates AS (
        SELECT b.concept, b.x, b.y, b.width AS bw, b.height AS bh,
               i.thumbnail_s3_key, i.width AS iw, i.height AS ih,
               row_number() OVER (PARTITION BY b.concept ORDER BY b.width * b.height DESC) AS rn
        FROM boxes b
        JOIN images i USING (image_uuid)
        WHERE b.concept IN (SELECT concept FROM top_concepts)
          AND b.review_state = 'VERIFIED'
          AND NOT coalesce(b.truncated, false)
          AND NOT coalesce(b.occluded, false)
    )
    SELECT * FROM candidates WHERE rn = 1 ORDER BY concept
""").df()


def get_thumbnail(key):
    return Image.open(io.BytesIO(s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()))


with ThreadPoolExecutor(max_workers=8) as pool:
    thumbs = list(pool.map(get_thumbnail, gallery.thumbnail_s3_key))

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, row, thumb in zip(axes.flat, gallery.itertuples(), thumbs):
    s = thumb.width / row.iw
    ax.imshow(thumb)
    ax.add_patch(
        Rectangle(
            (row.x * s, row.y * s),
            row.bw * s,
            row.bh * s,
            fill=False,
            edgecolor="yellow",
            linewidth=2,
        )
    )
    ax.set_title(row.concept, fontsize=10)
    ax.axis("off")
for ax in axes.flat[len(gallery) :]:
    ax.axis("off")
plt.tight_layout()
plt.show()

Next, where the images come from. Not every image has a position or depth, so we keep those that do.
Color shows depth.

In [ ]:
geo = con.sql("""
    SELECT longitude, latitude, depth_meters
    FROM images
    WHERE latitude IS NOT NULL AND longitude IS NOT NULL AND depth_meters IS NOT NULL
""").df()

fig, ax = plt.subplots(figsize=(14, 7))
sc = ax.scatter(geo.longitude, geo.latitude, c=geo.depth_meters, cmap="viridis_r", s=2, alpha=0.6)
fig.colorbar(sc, ax=ax, label="Depth (m)")
ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(f"FathomNet images with position and depth (n = {len(geo):,})")
ax.grid(True, linestyle="--", alpha=0.3)
plt.show()

### Q: What is one question that you have answered using these data? Can you show us how you came to that answer?

**Which organisms have enough training data, and how far does that extend into the deep?**

When we build detectors for ocean video, the first question is not which architecture to use but which
concepts we can learn at all. FathomNet's label distribution is strongly long-tailed: a few
concepts have tens of thousands of boxes, and most have only a handful. The deep sea compounds this,
because sampling effort drops with depth. Here we measure both, and then build a reproducible,
license-aware training subset without downloading the whole dataset.

First, the long tail. We count verified boxes per concept and plot them by rank on log axes.

In [ ]:
per_concept = con.sql("""
    SELECT concept, count(*) AS n
    FROM boxes
    WHERE review_state = 'VERIFIED'
    GROUP BY concept
    ORDER BY n DESC
""").df()

fig, ax = plt.subplots(figsize=(10, 6))
ax.loglog(range(1, len(per_concept) + 1), per_concept.n, marker=".", linestyle="none")
for threshold in [100, 1000]:
    ax.axhline(threshold, color="gray", linestyle="--", linewidth=1)
    ax.text(1, threshold * 1.1, f"{threshold} boxes", color="gray")
ax.set_xlabel("Concept rank")
ax.set_ylabel("Verified boxes")
ax.set_title("Verified boxes per concept")
plt.show()

for threshold in [10, 100, 1000]:
    n = (per_concept.n >= threshold).sum()
    print(f"{n:5d} of {len(per_concept)} concepts have >= {threshold} verified boxes")

Next, how the number of trainable concepts changes with depth. We bin boxes into depth bands and count,
in each band, how many concepts have at least 100 verified boxes.

In [ ]:
bands = con.sql("""
    WITH binned AS (
        SELECT b.concept,
               CASE
                   WHEN i.depth_meters < 200 THEN '0-200 m'
                   WHEN i.depth_meters < 1000 THEN '200-1000 m'
                   WHEN i.depth_meters < 2000 THEN '1000-2000 m'
                   WHEN i.depth_meters < 4000 THEN '2000-4000 m'
                   ELSE '4000+ m'
               END AS band,
               CASE
                   WHEN i.depth_meters < 200 THEN 0
                   WHEN i.depth_meters < 1000 THEN 1
                   WHEN i.depth_meters < 2000 THEN 2
                   WHEN i.depth_meters < 4000 THEN 3
                   ELSE 4
               END AS band_order
        FROM boxes b JOIN images i USING (image_uuid)
        WHERE b.review_state = 'VERIFIED' AND i.depth_meters IS NOT NULL
    ),
    counts AS (
        SELECT band, band_order, concept, count(*) AS n FROM binned GROUP BY ALL
    )
    SELECT band, band_order,
           sum(n) AS n_boxes,
           count(*) AS n_concepts,
           count(*) FILTER (WHERE n >= 100) AS n_trainable
    FROM counts
    GROUP BY band, band_order
    ORDER BY band_order
""").df()

fig, ax1 = plt.subplots(figsize=(10, 6))
ax1.bar(bands.band, bands.n_trainable, color="#3498db", label="Concepts with >= 100 verified boxes")
ax1.set_ylabel("Concepts with >= 100 verified boxes")
ax2 = ax1.twinx()
ax2.plot(bands.band, bands.n_boxes, color="black", marker="o", label="Verified boxes")
ax2.set_ylabel("Verified boxes")
ax1.set_title("Trainable concepts by depth band")
plt.show()

bands

Finally, turning the answer into a training set. Suppose we want a detector for concepts with at
least 1000 verified boxes, restricted to images and annotations we can use commercially (CC0 or CC-BY).
The selection is a query against a pinned release, so anyone can reproduce it exactly. We write a COCO
file and download only the matching images.

Here we cap the download at 50 images to keep the example quick. To scale up, remove the cap and run on
an Amazon EC2 or Amazon SageMaker instance in `us-west-2`, where reads from the bucket are fastest.

In [ ]:
OPEN_LICENSES = ["CC0-1.0", "CC-BY-4.0"]
MIN_BOXES = 1000
MAX_IMAGES = 50
OUT = Path("fathomnet-subset")

subset = con.execute(
    """
    WITH eligible AS (
        SELECT b.*, i.s3_key, i.width AS image_width, i.height AS image_height
        FROM boxes b JOIN images i USING (image_uuid)
        WHERE b.review_state = 'VERIFIED'
          AND list_contains($licenses, i.image_license)
          AND list_contains($licenses, b.annotation_license)
    ),
    common AS (
        SELECT concept FROM eligible GROUP BY concept HAVING count(*) >= $min_boxes
    ),
    chosen AS (
        SELECT DISTINCT image_uuid FROM eligible WHERE concept IN (SELECT concept FROM common)
        ORDER BY image_uuid
        LIMIT $max_images
    )
    SELECT e.* FROM eligible e
    WHERE e.image_uuid IN (SELECT image_uuid FROM chosen)
      AND e.concept IN (SELECT concept FROM common)
""",
    {"licenses": OPEN_LICENSES, "min_boxes": MIN_BOXES, "max_images": MAX_IMAGES},
).df()

print(
    f"{subset.image_uuid.nunique()} images, {len(subset)} boxes, {subset.concept.nunique()} concepts"
)

In [ ]:
categories = {c: i for i, c in enumerate(sorted(subset.concept.unique()), start=1)}
images_df = subset.drop_duplicates("image_uuid").reset_index(drop=True)
image_ids = {u: i for i, u in enumerate(images_df.image_uuid, start=1)}

coco = {
    "info": {"description": f"FathomNet subset, release {RELEASE}"},
    "categories": [{"id": i, "name": c} for c, i in categories.items()],
    "images": [
        {
            "id": image_ids[r.image_uuid],
            "file_name": Path(r.s3_key).name,
            "width": int(r.image_width),
            "height": int(r.image_height),
        }
        for r in images_df.itertuples()
    ],
    "annotations": [
        {
            "id": i,
            "image_id": image_ids[r.image_uuid],
            "category_id": categories[r.concept],
            "bbox": [int(r.x), int(r.y), int(r.width), int(r.height)],
            "area": int(r.width) * int(r.height),
            "iscrowd": 0,
        }
        for i, r in enumerate(subset.itertuples(), start=1)
    ],
}

(OUT / "images").mkdir(parents=True, exist_ok=True)
(OUT / "annotations.json").write_text(json.dumps(coco))


def download(key):
    dest = OUT / "images" / Path(key).name
    if not dest.exists():
        s3.download_file(BUCKET, key, str(dest))


with ThreadPoolExecutor(max_workers=16) as pool:
    list(pool.map(download, images_df.s3_key))

print(f"Wrote {OUT / 'annotations.json'} and {len(images_df)} images")

The result is a standard COCO dataset that any detection framework can train on. Because the
selection is pinned to a release and defined entirely by the query, the same code yields the same
dataset next year, even after FathomNet has grown.

### Q: What is one unanswered question that you think could be answered using these data? Do you have any recommendations or advice for someone wanting to answer this question?

**Can we learn to find every organism in an image when most of them were never labeled?**

FathomNet annotations are rarely exhaustive. Each collection was annotated for the taxa its contributors
cared about, so an image labeled with one fish may contain ten unlabeled sea stars. Standard detector
training treats those unlabeled animals as background and learns to ignore them. This is a
positive-unlabeled (PU) learning problem, and a major obstacle to a general-purpose "find all the life in
this frame" model for the ocean. It was the focus of the
[FathomNetCLEF 2026](https://www.kaggle.com/competitions/fathomnet-2026) challenge, and remains open.

Some starting points:

- **Evaluate on fully annotated images.** The
  [full-coverage test sets](https://huggingface.co/datasets/FathomNet/full-coverage-test-sets) list
  FathomNet image UUIDs in which every visible organism is annotated. Join them to the `images` table to
  get a clean evaluation set.
- **Use the taxonomy.** Labels sit at mixed ranks. Resolve them to a lineage with FathomNet's
  [WoRMS name server](https://database.fathomnet.org/worms/docs) to train hierarchically (for example,
  predict phylum confidently even when species is uncertain) and avoid penalizing a correct coarse
  answer.
- **Lean on foundation models.** Class-agnostic proposals from segmentation models such as SAM 3, or
  embeddings from biological models such as BioCLIP 2, can flag likely-unlabeled organisms for
  pseudo-labeling or exclusion from the negative set.
- **Test for domain shift.** Hold out entire institutions or imaging platforms rather than random images
  (`collections.owner_institution_code` via `image_collections`, or `images.imaging_type`). A
  model that only works on the camera it was trained on is not yet useful to the ocean community.
- **Give back.** New annotations and validated pseudo-labels can be contributed to the
  [FathomNet Database](https://database.fathomnet.org), and shared models are welcome on
  [Hugging Face](https://huggingface.co/FathomNet).